# Exercise 3: Explore the Databricks Workspace
Goal: get familiar with the Databricks Free Edition workspace and confirm that the three tables are loaded correctly.
- Created the folder Lesson01_Bala and a Python notebook on serverless compute.
- Explored the left sidebar: Workspace (folders and notebooks), Catalog (catalogs, schemas, tables), Compute (serverless compute), Workflows (jobs and pipelines), Data Ingestion (loading data).
- Confirmed the 3 tables in Catalog: salesorderdetail, product, salesterritory.
- DESCRIBE on each table to confirm the columns are correctly named, not _c0, _c1.
- COUNT of rows in each table.

In [0]:
display(spark.sql("DESCRIBE adventureworks.bronze.salesorderdetail"))

In [0]:
display(spark.sql("DESCRIBE adventureworks.bronze.product"))

In [0]:
display(spark.sql("DESCRIBE adventureworks.bronze.salesterritory"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS rows FROM adventureworks.bronze.salesorderdetail"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS rows FROM adventureworks.bronze.product"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS rows FROM adventureworks.bronze.salesterritory"))

# Exercise 4: Verify the Star Before You Query It
Goal: check that the fact table and the dimension tables are consistent before running analysis.
- Total rows vs distinct SalesOrderDetailID in salesorderdetail: both 121,317, so the ID never repeats.
- Row counts: salesorderdetail 121,317 (fact table), product 504 and salesterritory 10 (dimensions). Plausible for a fact table vs two small dimensions.
- LEFT JOIN salesorderdetail to product: 121,317 rows, unchanged, so every key matches and there is no fan-out.
- RIGHT JOIN: 121,555 rows. The extra 238 rows are products that were never sold.
- Duplicate key check on product (GROUP BY ... HAVING COUNT > 1): no rows, so no duplicate ProductID.
- salesorderdetail has no TerritoryID column, so it cannot be joined to salesterritory directly.

In [0]:
display(spark.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT SalesOrderDetailID) AS distinct_ids FROM adventureworks.bronze.salesorderdetail"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS rows_after_join FROM adventureworks.bronze.salesorderdetail f LEFT JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS rows_right_join FROM adventureworks.bronze.salesorderdetail f RIGHT JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID"))

## Join check
LEFT JOIN fact to product: 121,317 rows (same as fact), so no duplicate ProductIDs and no fan-out.
RIGHT JOIN: 121,555 rows. The extra 238 rows are products that were never sold; a RIGHT JOIN keeps all products, a LEFT JOIN keeps all sales.

In [0]:
display(spark.sql("SELECT ProductID, COUNT(1) AS n FROM adventureworks.bronze.product GROUP BY ProductID HAVING COUNT(1) > 1"))

# Exercise 5: Your First SQL Queries
Goal: run the first SQL queries on the tables.
- Top 10 newest rows of salesorderdetail by ModifiedDate (newest are 2025-06-29).
- Total rows 121,317 and 266 distinct ProductID sold, compared with 504 products in the product table (238 never sold).
- Top 10 revenue products: Mountain-200 bikes lead, Mountain-200 Black 38 is first with 4,400,592.80.
- NULL or negative LineTotal: 0 rows, so nothing to clean.
- Top 5 revenue products shown as a bar chart.

In [0]:
display(spark.sql("SELECT SalesOrderID, SalesOrderDetailID, ProductID, OrderQty, LineTotal, ModifiedDate FROM adventureworks.bronze.salesorderdetail ORDER BY ModifiedDate DESC LIMIT 10"))

In [0]:
display(spark.sql("SELECT COUNT(1) AS total_rows, COUNT(DISTINCT ProductID) AS distinct_products FROM adventureworks.bronze.salesorderdetail"))

In [0]:
display(spark.sql("SELECT f.ProductID, p.Name, ROUND(SUM(f.LineTotal),2) AS revenue FROM adventureworks.bronze.salesorderdetail f JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID GROUP BY f.ProductID, p.Name ORDER BY revenue DESC LIMIT 10"))

In [0]:
display(spark.sql("SELECT COUNT(1) AS bad_rows FROM adventureworks.bronze.salesorderdetail WHERE LineTotal IS NULL OR LineTotal < 0"))

In [0]:
display(spark.sql("SELECT p.Name, ROUND(SUM(f.LineTotal),2) AS revenue FROM adventureworks.bronze.salesorderdetail f JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID GROUP BY p.Name ORDER BY revenue DESC LIMIT 5"))

Databricks visualization. Run in Databricks to view.

# Exercise 6: Data Profiling in SQL
Goal: profile the three tables: counts, distinct keys, orphan keys, most ordered products and revenue by product line.

In [0]:
display(spark.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT SalesOrderDetailID) AS distinct_keys FROM adventureworks.bronze.salesorderdetail"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT ProductID) AS distinct_keys FROM adventureworks.bronze.product"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT TerritoryID) AS distinct_keys FROM adventureworks.bronze.salesterritory"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS orphan_rows FROM adventureworks.bronze.salesorderdetail f LEFT JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID WHERE p.ProductID IS NULL"))

In [0]:
display(spark.sql("SELECT COUNT(*) AS never_sold FROM adventureworks.bronze.product p LEFT JOIN adventureworks.bronze.salesorderdetail f ON p.ProductID = f.ProductID WHERE f.ProductID IS NULL"))

In [0]:
display(spark.sql("SELECT f.ProductID, p.Name, COUNT(*) AS times_ordered, SUM(f.OrderQty) AS total_qty FROM adventureworks.bronze.salesorderdetail f JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID GROUP BY f.ProductID, p.Name ORDER BY times_ordered DESC LIMIT 10"))

In [0]:
display(spark.sql("SELECT p.ProductLine, ROUND(SUM(f.LineTotal),2) AS revenue FROM adventureworks.bronze.salesorderdetail f JOIN adventureworks.bronze.product p ON f.ProductID = p.ProductID GROUP BY p.ProductLine ORDER BY revenue DESC"))

Databricks visualization. Run in Databricks to view.

## Exercise 6 findings
- Key check: salesorderdetail 121,317 rows = 121,317 distinct SalesOrderDetailID; product 504 = 504 distinct ProductID; salesterritory 10 = 10 distinct TerritoryID. All keys are unique.
- Orphaned ProductID in sales: 0. Every sale row has a matching product, so the join is safe.
- Products never sold: 238 of 504. They exist in the product table but have no sales rows, so a RIGHT JOIN or reverse LEFT JOIN returns them with NULL sales.
- Most frequently ordered: Water Bottle 30 oz (4,688 orders), AWC Logo Cap (3,382), Patch Kit (3,354). Frequent items are low-priced accessories.
- Revenue by product line: R (Road) 48.30M, M (Mountain) 42.46M, T (Touring) 16.01M, S (Standard) 2.54M, no product line 0.54M. Road bikes drive the most revenue.
- Insight: the most ordered products are cheap accessories, but the highest revenue comes from bikes.